In [2]:
import pandas as pd 
df=pd.read_csv("monitor_data.csv")
df.head()

,Time,Voltage(V),Current(A),Power(W),Leakage Current(mA),total_forward_energy(kWh),price(DZD)
0,2025-08-12 22:30:01,355.92,7.68,24.2,0,16.6,88.5372
1,2025-08-12 22:30:11,343.12,7.68,22.2,0,16.6,88.5372
2,2025-08-12 22:30:21,358.48,7.68,21.2,0,16.6,88.5372
3,2025-08-12 22:30:31,373.84,7.68,20.2,0,16.6,88.5372
4,2025-08-12 22:30:41,373.84,7.68,20.2,0,16.6,88.5372


In [3]:
import os
import time
import base64
import sounddevice as sd
import soundfile as sf
import whisper
import pyttsx3
from tuya_connector import TuyaOpenAPI

# ---------------- CONFIG ----------------
ACCESS_ID = "3rtkjvnvn7echqjc4ffn"
ACCESS_KEY = "93ad1b54d5544c64b5e893de55afab4a"
API_ENDPOINT = "https://openapi.tuyaeu.com"
DEVICE_ID = "bf0178074ad9c548aduc79"

MODEL_NAME = "base"   # whisper model size (tiny, base, small, medium, large)
AUDIO_FILE = "command.wav"

# Init Tuya Cloud
openapi = TuyaOpenAPI(API_ENDPOINT, ACCESS_ID, ACCESS_KEY)
openapi.connect()

# Init TTS
tts = pyttsx3.init()

def speak(text):
    print("[AGENT]:", text)
    tts.say(text)
    tts.runAndWait()

# ============= TUYA FUNCTIONS =============
def get_measurements():
    voltage = current = power = energy = leakage = None
    resp = openapi.get(f"/v1.0/devices/{DEVICE_ID}/status")
    for item in resp.get("result", []):
        if item["code"] == "phase_a":
            raw = base64.b64decode(item["value"])
            voltage = (raw[2] | (raw[3] << 8)) / 10.0
            current = (raw[4] | (raw[5] << 8)) / 100.0
            power   = (raw[6] | (raw[7] << 8)) / 10.0
        elif item["code"] == "total_forward_energy":
            energy = item["value"] / 100.0
        elif item["code"] == "leakage_current":
            leakage = item["value"]

    return voltage, current, power, energy, leakage

def set_switch(state: bool):
    commands = {'commands': [{'code': 'switch', 'value': state}]}
    openapi.post(f"/v1.0/devices/{DEVICE_ID}/commands", commands)
    speak(f"Breaker turned {'ON' if state else 'OFF'}.")

# ============= SPEECH FUNCTIONS =============
def record_audio(filename, duration=4, samplerate=16000):
    """Record voice from mic"""
    speak("Listening...")
    recording = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype="int16")
    sd.wait()
    sf.write(filename, recording, samplerate)
    return filename

def transcribe_audio(filename):
    """Transcribe voice with whisper"""
    model = whisper.load_model(MODEL_NAME)
    result = model.transcribe(filename)
    return result["text"].strip().lower()

# ============= HANDLER =============
def handle_query(text):
    if "turn on" in text:
        set_switch(True)
    elif "turn off" in text:
        set_switch(False)
    elif "status" in text or "data" in text:
        v,i,p,e,l = get_measurements()
        speak(f"Voltage {v} volts, Current {i} amps, Power {p} watts, Energy {e} kilowatt hours, Leakage {l} milliamps.")
    else:
        speak("Sorry, I did not understand.")

# ============= MAIN LOOP =============
if __name__ == "__main__":
    speak("Voice agent is ready.")
    while True:
        input("Press Enter to give a command...")
        audio = record_audio(AUDIO_FILE)
        text = transcribe_audio(audio)
        print("audio:",audio)
        print("[YOU]:", text)
        if text:
            handle_query(text)

[AGENT]: Voice agent is ready.
[AGENT]: Listening...


FileNotFoundError: [WinError 2] Le fichier spécifié est introuvable